In [1]:
## STEP 0: Need Some Packages et al.

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
# STEP 1: LOAD your Data
# NOTE: Your data needs to be UPLOADED
# Click the little "Folder" icon on the left side
# Use the "Upload Button"

# Load the data
data_price = pd.read_csv('Data_Price_Python.csv')

In [3]:
# STEP 2: EXPLORE the Data (does it look like what you expected to have?)
# Explore the data (what variables are in it and what are some basic statistics for these variables)

print(data_price.info())
print(data_price.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   WEEK    75 non-null     int64  
 1   UNITS1  75 non-null     int64  
 2   UNITS2  75 non-null     int64  
 3   UNITS3  75 non-null     int64  
 4   UNITS4  75 non-null     int64  
 5   PRICE1  75 non-null     float64
 6   FEAT1   75 non-null     int64  
 7   DISP1   75 non-null     float64
 8   PRICE2  75 non-null     float64
 9   FEAT2   75 non-null     int64  
 10  DISP2   75 non-null     float64
 11  PRICE3  75 non-null     float64
 12  PRICE4  75 non-null     float64
 13  FEAT4   75 non-null     int64  
dtypes: float64(6), int64(8)
memory usage: 8.3 KB
None
            WEEK       UNITS1       UNITS2      UNITS3      UNITS4     PRICE1  \
count  75.000000    75.000000    75.000000   75.000000   75.000000  75.000000   
mean   38.000000   234.226667   280.226667   72.653333   52.186667   3.642925   
std    

In [4]:
# STEP 3: PREPARE DATA needed for the model
# In our case we have to CREATE 2 NEW variables: log(Units) and log(Prices)

# Create log UNITS
data_price['logUNITS1'] = np.log(data_price['UNITS1'])
data_price['logUNITS2'] = np.log(data_price['UNITS2'])
data_price['logUNITS3'] = np.log(data_price['UNITS3'])
data_price['logUNITS4'] = np.log(data_price['UNITS4'])

# Create log PRICES
data_price['logPRICE1'] = np.log(data_price['PRICE1'])
data_price['logPRICE2'] = np.log(data_price['PRICE2'])
data_price['logPRICE3'] = np.log(data_price['PRICE3'])
data_price['logPRICE4'] = np.log(data_price['PRICE4'])

In [5]:
# STEP 4: RUN the Model(s)
# Define y (Dependent Variable). For this example, Sales for Brand 1 (UNITS1) is used.
# Define X (Independent Variables). For this example, OWN Marketing Activity (PRICE1, FEAT1 and DISP1) is used
# Decide on Model to ue. For example, a linear model where Sales for Brand 1 (UNITS1) are a function of Units1's Price (PRICE1), Feature (FEAT1), and Display (DISP1)

# Models from Slide "PRICE REGRESSIONS – EXAMPLE OREO"
# These are the basic Models that only have the Price of Oreo
lm_linear   = sm.OLS(data_price['UNITS1'], sm.add_constant(data_price[['PRICE1']])).fit()
lm_semi_log = sm.OLS(data_price['logUNITS1'], sm.add_constant(data_price[['PRICE1']])).fit()
lm_log_log  = sm.OLS(data_price['logUNITS1'], sm.add_constant(data_price[['logPRICE1']])).fit()


# Models from Slide "INCORPORATING PROMOTIONAL RESPONSE", results are on the next slide in the deck
# These models have Oreo's own Price and also Feature and Display for Oreo
# Note we cannot run the log-log model as Feature and Display include zeros so we only run two models
lm_linear_own   = sm.OLS(data_price['UNITS1'], sm.add_constant(data_price[['PRICE1', 'FEAT1', 'DISP1']])).fit()
lm_semi_log_own = sm.OLS(data_price['logUNITS1'], sm.add_constant(data_price[['PRICE1', 'FEAT1', 'DISP1']])).fit()


# Models from Slide "CROSS-PRICE EFFECTS AND CROSS-PROMOTION EFFECTS", results are on the next slide in the deck
# These models have Oreo's own Marketing activity (Price, Feature and Display) and ALL competitive marketing activity
# Note we cannot run the log-log model as Feature and Display include zeros so we only run two models
# Note that there are no Display and Feature for Brand 3 and no Feature for Brand 4
lm_linear_own_comp   = sm.OLS(data_price['UNITS1'], sm.add_constant(data_price[['PRICE1', 'FEAT1', 'DISP1', 'PRICE2', 'FEAT2', 'DISP2', 'PRICE3', 'PRICE4', 'FEAT4']])).fit()
lm_semi_log_own_comp = sm.OLS(data_price['logUNITS1'], sm.add_constant(data_price[['PRICE1', 'FEAT1', 'DISP1', 'PRICE2', 'FEAT2', 'DISP2', 'PRICE3', 'PRICE4', 'FEAT4']])).fit()

In [6]:
# Display Results
print(lm_linear.summary())
print(lm_semi_log.summary())
print(lm_log_log.summary())
print(lm_linear_own.summary())
print(lm_semi_log_own.summary())
print(lm_linear_own_comp.summary())
print(lm_semi_log_own_comp.summary())

                            OLS Regression Results                            
Dep. Variable:                 UNITS1   R-squared:                       0.097
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     7.860
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00647
Time:                        03:20:26   Log-Likelihood:                -579.15
No. Observations:                  75   AIC:                             1162.
Df Residuals:                      73   BIC:                             1167.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2163.0620    690.977      3.130      0.0

In [7]:
# STEP 5: Managerially Useful "THINGS". In our case here we want to caculate the PRICE Elasticity of Demand

# Calculate Price Elasticity from the Linear Model that has only Price
# Note: Can naturally do this for ALL models that have Price in them
price_coeff_linear = lm_linear.params['PRICE1']
elas_linear = price_coeff_linear * data_price['PRICE1'].mean() / data_price['UNITS1'].mean()
print("Price Elasticity (Linear Model):", elas_linear)

Price Elasticity (Linear Model): -8.234909389934359
